# W02d - Predictions with ML Regression (Full Custom Input)

In [1]:
### Import necessary libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import random
import pickle
import time
from text_operations import manipulate_text, translate_to_english
from dataset_operations import get_title_code, get_title_snr_code, get_position_code, get_position_snr_code
from dataset_operations import get_emp_type_coef, get_highest_edu_degree_coef, get_edu_degree_major_code
from dataset_operations import get_company_code, get_main_skills_code
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import accuracy_score, f1_score, mean_squared_error, mean_absolute_error, r2_score
from sklearn.model_selection import cross_val_score, cross_validate, GridSearchCV
import xgboost
warnings.filterwarnings('ignore')
pd.set_option('display.max_columns',None)
num_topics = 25

In [2]:
### Load the salary dataset with the correct date, plus the country coefficient and feature codes/coefs dataset
salaries = pd.read_csv('Salaries_v4_202412161100_enhanced_tm.csv')
salaries = salaries.drop('Unnamed: 0', axis=1)
countries_coef = pd.read_csv('countries_salary_coef.csv')
registered_coefs = pd.read_csv('features_codes_coefs.csv')
registered_coefs = registered_coefs.drop('Unnamed: 0', axis=1)

## Dataset Information

In [3]:
### Number of rows (salaries) and columns
print("SALARY DATASET SHAPE -> ROWS: {}  COLUMNS: {}".format(salaries.shape[0], salaries.shape[1]))

SALARY DATASET SHAPE -> ROWS: 19554  COLUMNS: 40


In [4]:
### First few rows of the dataset
salaries.head()

,id,found_country,title,position,employment_type,company,company_score,edu_degrees,edu_degrees_major,working_year,education_score,ms_counts,skill_counts,main_skills,skills,amount_usd,country_coef,title_code,title_slr_coef,title_seniority,title_seniority_slr_coef,position_code,position_slr_coef,position_seniority,position_seniority_slr_coef,employment_type_coef,emp_type,employment_type_slr_coef,highest_edu_deg_name,highest_edu_deg_lvl,edu_deg_slr_coef,edu_degree_major_code,edu_degree_major_slr_coef_avg,edu_degree_major_slr_coef_max,company_code,company_slr_coef,main_skill_code,main_skill_slr_coef_avg,main_skill_slr_coef_max,topic_model_no
0,0000083c-7054-4a2b-b675-6ac664c66bfb,United States,"Software Developer II at Audible, Inc.",Software Developer II,Full-time,"Audible, Inc.",8.9,"HIGH_SCHOOL,MASTERS,UNDERGRADUATE","Bachelor’s Degree, Computer Science,High Schoo...",11,8.1,21,20,"Ajax, C, C++, CSS, Data Engineering, HTML, HTM...","ajax, c, c++, cascading style sheets (css), cs...",192000,1.00,WC68,0.488,NO_SENIORITY,0.488,WC66,0.492,NO_SENIORITY,0.518,0.99,FULL_TIME,1.0,MASTERS,0.75,0.770,EDM17,0.796,0.796,UNCT,0.368,"MS01,MS02,MS03,MS04,MS05,MS06,MS07,MS11,MS17,M...",0.872,0.944,5
1,00013847-ecf1-4a5a-ba44-16475dc28eba,United States,Retail Associate at Converse,Retail Associate,Full-time,NaN,NaN,UNDERGRADUATE,NaN,5,NaN,14,10,"Communication, Customer Service, Deadline Mana...","communication, customer service, employee trai...",38000,1.00,UNCT,0.319,ASSOCIATE,0.319,UNCT,0.329,ASSOCIATE,0.340,0.99,FULL_TIME,1.0,UNDERGRADUATE,0.50,0.657,NO_DEGREE,0.200,0.200,NO_COMPANY,0.200,"MS00,MS13,MS15,MS16,MS28,MS30,MS31,MS33,MS34,M...",0.646,0.731,0
2,00018332-5b5d-4c23-88f8-1c2cdc133e28,United Kingdom,Test Engineer at Sky,Test Engineer,Full-time,NaN,NaN,UNDERGRADUATE,"Bachelor of Science (BSc), Computer Software E...",12,7.0,15,20,"Agile, Attention to Detail, Automated Testing,...","agile methodologies, api testing, communicatio...",66191,0.61,WC76,0.399,NO_SENIORITY,0.488,WC67,0.400,NO_SENIORITY,0.518,0.99,FULL_TIME,1.0,UNDERGRADUATE,0.50,0.657,"EDM14,EDM30,EDM38,EDM53",0.779,0.823,NO_COMPANY,0.200,"MS00,MS16,MS40,MS42,MS43,MS45,MS62",0.774,0.844,21
3,000c1054-ab28-4c4d-90b0-fa4b1ed31a2a,United States,Hardware Engineer at Google,Hardware Engineer,NaN,Google,8.7,MASTERS,"Master of Science (MS), Computer Engineering",13,8.9,9,20,"C, Circuit Design, Debugging, Embedded Softwar...","asic, c, computer architecture, debugging, emb...",341000,1.00,WC36,0.955,NO_SENIORITY,0.488,WC32,0.959,NO_SENIORITY,0.518,0.10,NO_EMP_TYPE,0.2,MASTERS,0.75,0.770,EDM14,0.823,0.823,CMP01,0.845,"MS20,MS42,MS43",0.873,0.944,17
4,00145b03-e286-4bdc-9063-ed5d2095306a,United States,BI Engineer @ Amazon | MS,BIE II,NaN,Amazon,8.4,MASTERS,"Master of Science - MS, Data Analytics",3,8.5,6,9,"JavaScript, Marketing Research, Microsoft Exce...","javascript, microsoft excel, microsoft office,...",185000,1.00,WC07,0.503,NO_SENIORITY,0.488,WC06,0.516,NO_SENIORITY,0.518,0.10,NO_EMP_TYPE,0.2,MASTERS,0.75,0.770,EDM19,0.655,0.655,CMP00,0.593,"MS03,MS08,MS12,MS14,MS19",0.769,0.870,24


In [5]:
### All features' info in the salary dataset
salaries.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 19554 entries, 0 to 19553
Data columns (total 40 columns):
 #   Column                         Non-Null Count  Dtype  
---  ------                         --------------  -----  
 0   id                             19554 non-null  object 
 1   found_country                  19554 non-null  object 
 2   title                          19542 non-null  object 
 3   position                       19554 non-null  object 
 4   employment_type                12397 non-null  object 
 5   company                        16936 non-null  object 
 6   company_score                  15739 non-null  float64
 7   edu_degrees                    19000 non-null  object 
 8   edu_degrees_major              18623 non-null  object 
 9   working_year                   19554 non-null  int64  
 10  education_score                14956 non-null  float64
 11  ms_counts                      19554 non-null  int64  
 12  skill_counts                   19554 non-null 

In [6]:
### Stats for numerical features of the dataset
salaries.describe()

,company_score,working_year,education_score,ms_counts,skill_counts,amount_usd,country_coef,title_slr_coef,title_seniority_slr_coef,position_slr_coef,position_seniority_slr_coef,employment_type_coef,employment_type_slr_coef,highest_edu_deg_lvl,edu_deg_slr_coef,edu_degree_major_slr_coef_avg,edu_degree_major_slr_coef_max,company_slr_coef,main_skill_slr_coef_avg,main_skill_slr_coef_max,topic_model_no
count,15739.000000,19554.000000,14956.000000,19554.000000,19554.000000,19554.000000,19554.000000,19554.000000,19554.000000,19554.000000,19554.000000,19554.000000,19554.000000,19554.000000,19554.000000,19554.000000,19554.000000,19554.000000,19554.000000,19554.000000,19554.000000
mean,8.248993,11.449882,8.401484,15.738008,18.777948,163366.552981,0.966054,0.452360,0.511637,0.456774,0.550829,0.651904,0.701566,0.612090,0.705434,0.695443,0.714984,0.517684,0.784424,0.884132,12.980158
std,0.561583,7.213469,1.383108,7.306042,7.814769,81118.159433,0.109942,0.145708,0.074651,0.148300,0.087546,0.426533,0.382863,0.178708,0.128333,0.147528,0.158501,0.206434,0.083624,0.089182,7.183227
min,3.800000,0.000000,1.600000,1.000000,1.000000,21000.000000,0.610000,0.102000,0.200000,0.115000,0.248000,0.100000,0.200000,0.100000,0.200000,0.200000,0.200000,0.181000,0.569000,0.569000,0.000000
25%,8.100000,6.000000,7.500000,11.000000,15.000000,102000.000000,1.000000,0.350000,0.488000,0.338000,0.518000,0.100000,0.200000,0.500000,0.657000,0.591000,0.603000,0.368000,0.712000,0.833000,6.000000
50%,8.400000,10.000000,8.600000,16.000000,20.000000,150000.000000,1.000000,0.484000,0.488000,0.492000,0.518000,0.990000,1.000000,0.500000,0.657000,0.732000,0.749000,0.541000,0.801000,0.894000,14.000000
75%,8.600000,15.000000,9.700000,20.000000,20.000000,210000.000000,1.000000,0.539000,0.488000,0.565000,0.518000,0.990000,1.000000,0.750000,0.770000,0.796000,0.806000,0.676000,0.856000,0.964000,19.000000
max,10.000000,57.000000,10.000000,56.000000,82.000000,560000.000000,1.000000,1.000000,1.000000,1.000000,1.000000,0.990000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,24.000000


In [7]:
### First few lines of country names and coefficients
countries_coef.head(10)

,tr_name,en_name,salary_coefficient
0,Amerika Birleşik Devletleri,United States,1.000
1,Kanada,Canada,0.769
2,Monako,Monaco,0.766
3,Lüksemburg,Luxembourg,0.766
4,İrlanda,Ireland,0.765
5,İsrail,Israel,0.763
6,İzlanda,Iceland,0.750
7,İsviçre,Switzerland,0.740
8,Malta,Malta,0.735
9,Tayland,Thailand,0.729


In [8]:
### First few lines of feature codes and coefficients
registered_coefs.head(10)

,feature,code,coef
0,title,WC00,0.607
1,title,WC01,0.477
2,title,WC02,0.484
3,title,WC03,0.697
4,title,WC04,0.797
5,title,WC05,0.481
6,title,WC06,0.350
7,title,WC07,0.503
8,title,WC08,0.320
9,title,WC09,0.318


### Set a Full Custom Input and Get Necessary Numerical Values of Every Feature

In [11]:
### Remember which features are required in order to be eligible for prediction with ML models
# company_score, working_year, education_score, ms_counts, skill_counts, 
# country_coef, title_slr_coef, title_seniority_slr_coef, position_slr_coef, position_seniority_slr_coef, 
# employment_type_coef, employment_type_slr_coef, highest_edu_deg_lvl, edu_deg_slr_coef, edu_degree_major_slr_coef_max, 
# company_slr_coef, main_skill_slr_coef_avg

### COMPANY SCORE (5.0-10.0)
# company_score = 8.4
company_score = random.randint(50,100) / 10
### WORKING YEAR (0-50)
# working_year = 16
working_year = random.randint(0,50)
### EDUCAITON SCORE (5.0-10.0)
# education_score = 8.7
education_score = random.randint(50,100) / 10
### MAIN SKILL COUNT (1-20)
# ms_counts = 12
ms_counts = random.randint(1,20)
### SKILL COUNT (1-50)
# skill_counts = 28
skill_counts = random.randint(ms_counts,50)
### COUNTRY COEFFICIENT (0.0-1.0)
country_list = ['Argentina', 'Australia', 'Austria', 'Bangladesh', 'Belarus', 'Belgium', 'Brazil', 'Canada', 
    'China', 'Denmark', 'Egypt', 'Estonia', 'Finland', 'France', 'Germany', 'Iceland', 'India', 'Indonesia', 'Ireland', 
    'Israel', 'Italy', 'Japan', 'Luxembourg', 'Mexico', 'Netherlands', 'New Zealand', 'Norway', 'Pakistan', 'Philippines', 
    'Poland', 'Romania', 'Russia', 'Serbia', 'Singapore', 'Spain', 'South Korea', 'Sweden', 'Switzerland', 'Turkey', 
    'Ukraine', 'United Kingdom', 'United States', 'Vietnam']
selected_country = country_list[random.randint(0,len(country_list)-1)]
country_coef = countries_coef[countries_coef['en_name'] == selected_country]['salary_coefficient'].values[0]
### TITLE SALARY COEFFICIENT (0.0-1.0)
title_list = ['Software Developer', 'Retail Associate', 'Test Engineer', 'Hardware Engineer', 'BI Engineer', 'Data Architect',
    'Data Analyst', 'Nurse', 'Principal Software Engineer', 'Sales Engineer', 'Networks Engineer', 'Senior User Experience Designer',
    'Technician', 'Quality Assurance Engineer', 'Warehouse Supervisor', 'School Counselor', 'Teacher', 'Senior Software Engineer',
    'Insurance Agent', 'Senior QA Engineer', 'Real Estate Agent', 'Machine Learning Engineer', 'DevOps Engineer', 'Corporate Counsel',
    'Visual Merchandiser', 'Investment Banking Analyst', 'Senior Financial Analyst', 'Senior Staff Data Scientist', 'Staff Accountant',
    'Business Analyst', 'Chemical Engineer', 'Beauty Advisor', 'Environmental Engineer', 'Senior Project Manager', 'Senior Data Analyst',
    'Program Coordinator', 'Frontend Engineer', 'Security Engineer', 'Staff Hardware Design Engineer', 'Transportation Planner',
    'Security Officer', 'Senior Data Scientist', 'Electrical Engineer', 'Actuary', 'Leasing Consultant', 'Junior Backend Developer',
    'Policy Analyst', 'Underwriter', 'AI Program Manager', 'Librarian', 'Urban Planner']
selected_title = title_list[random.randint(0,len(title_list)-1)]
title_code, _ = get_title_code(selected_title)
title_slr_coef = registered_coefs[(registered_coefs['feature'] == 'title') & \
        (registered_coefs['code'] == title_code)]['coef'].values[0]
### TITLE SENIORITY SALARY COEFFICIENT (0.0-1.0)
title_seniority_code = get_title_snr_code(selected_title)
title_seniority_slr_coef = registered_coefs[(registered_coefs['feature'] == 'title_snr') & \
        (registered_coefs['code'] == title_seniority_code)]['coef'].values[0]
### POSITION SALARY COEFFICIENT (0.0-1.0)
position_list = ['Software Developer', 'Test Engineer', 'Hardware Engineer', 'Data Architect', 'Data Analyst', 'Nurse',
    'Software Engineer', 'Technician', 'Sales Engineer', 'Network Engineer', 'Senior User Experience Designer',
    'Quality Assurance Engineer', 'Warehouse Supervisor', 'Teacher', 'Senior Software Engineer', 'Insurance Agent', 'Senior QA Engineer',
    'Corporate Counsel', 'DevOps Engineer', 'Visual Merchandiser', 'Investment Banking Analyst', 'Sr. Financial Analyst',
    'Staff Accountant', 'Data Scientist', 'Business Analyst', 'Beauty Advisor', 'Senior Project Manager', 'Program Coordinator',
    'SoC Design Engineer', 'Front End Engineer', 'Security Engineer', 'Senior Financial Analyst', 'Staff Hardware Design Engineer',
    'Transportation Planner', 'Help Desk Technician', ',Security Officer', 'Cloud Engineer', 'Electrical Engineer', 'Actuary',
    'Hardware Engineer', 'Machine Learning Engineer', 'Backend Developer', 'Policy Analyst', 'Underwriter', 'Project Manager',
    'Test Engineer', 'Data Analyst', 'System Administrator', 'Senior Data and Applied Scientist', 'Urban Planner', 'Paralegal',
    'Outside Sales Representative', 'Welder', 'UX Designer', 'Legal Secretary']
selected_position = position_list[random.randint(0,len(position_list)-1)]
position_code, _ = get_position_code(selected_position)
position_slr_coef = registered_coefs[(registered_coefs['feature'] == 'position') & \
        (registered_coefs['code'] == position_code)]['coef'].values[0]
### POSITION SENIORITY SALARY COEFFICIENT (0.0-1.0)
position_seniority_code = get_position_snr_code(selected_position)
position_seniority_slr_coef = registered_coefs[(registered_coefs['feature'] == 'position_snr') & \
        (registered_coefs['code'] == position_seniority_code)]['coef'].values[0]
### EMPLOYMENT TYPE COEFFICIENT (0.0-1.0)
emp_type_list = ['Full-time', 'Permanent Full-time', 'Permanent', 'Part-time', 'Contract', 'Self-employed', 'Freelance',
    'Apprenticeship', 'Internship', 'Seasonal']
selected_emp_type = emp_type_list[random.randint(0,len(emp_type_list)-1)]
employment_type_name, employment_type_coef = get_emp_type_coef(selected_emp_type)
### EMPLOYMENT TYPE SALARY COEFFICIENT (0.0-1.0)
employment_type_slr_coef = registered_coefs[(registered_coefs['feature'] == 'emp_type') & \
        (registered_coefs['code'] == employment_type_name)]['coef'].values[0]
### HIGHEST EDUCATION DEGREE LEVEL (0.0-1.0)
edu_degree_list = ['PHD', 'MASTERS', 'UNDERGRADUATE', 'ASSOCIATE', 'HIGH_SCHOOL']
edu_degree_amount = random.randint(1,3)
edu_degrees = ",".join(np.random.choice(edu_degree_list, edu_degree_amount, replace=False))
highest_edu_degree_name, highest_edu_degree_lvl = get_highest_edu_degree_coef(edu_degrees)
### EDUCATION DEGREE SALARY COEFFICIENT (0.0-1.0)
edu_deg_slr_coef = registered_coefs[(registered_coefs['feature'] == 'edu_degree') & \
        (registered_coefs['code'] == highest_edu_degree_name)]['coef'].values[0]
### EDUCATION DEGREE MAJOR SALARY COEFFICIENT (0.0-1.0) (AVG & MAX)
edu_degree_major_list = ['Computer Science', 'Computer Software Engineering', 'Information Technology', 'Mathematics', 'Data Analytics',
    'Artificial Intelligence', 'Electrical and Electronics Engineering', 'Nurse', 'Business Administration and Management', 'Finance',
    'Bachelor of Arts', 'Radio, Television and Film', 'Radiographer', 'Sociology and Educational Studies', 'Environmental Analysis', 
    'Psychology', 'Marketing', 'Physics', 'Mechanical Engineering', 'Fashion Merchandising', 'Financial Management', 'Law',
    'Chemical Engineering', 'Interior Design', 'Linguistics', 'Statistical Science', 'Urban Planning', 'Machine Learning', 'History',
    'Digital Media', 'Computer Networks', 'Business Analytics', 'Architect', 'Civil Engineering', 'Accounting']
edu_degree_majors = ", ".join(np.random.choice(edu_degree_major_list, edu_degree_amount, replace=False))
edu_degree_major_codes, _ = get_edu_degree_major_code(edu_degree_majors)
edu_degree_major_codes = edu_degree_major_codes.strip().replace(' ',',')
edu_degree_major_code_list = edu_degree_major_codes.split(',')
edu_degree_major_coefs = []
for x in range(len(edu_degree_major_code_list)):
    edu_degree_major_coefs.append(registered_coefs[(registered_coefs['feature'] == 'edu_degree_major') & \
                        (registered_coefs['code'] == edu_degree_major_code_list[x])]['coef'].values[0])
edu_degree_major_slr_coef_avg = round(np.array(edu_degree_major_coefs).mean(),3)
edu_degree_major_slr_coef_max = round(np.array(edu_degree_major_coefs).max(),3)
### COMPANY COEFFICIENT (0.0-1.0)
company_list = ['Amazon', 'Google', 'Oracle', 'Microsoft', 'Apple', 'Meta', 'Cisco', 'Amazon Web Services (AWS)', 'IBM',
    'JPMorganChase', 'LinkedIn', 'Capital One', 'Intel Corporation', 'SAIC', 'Tata Consultancy Services', 'Intuit', 'Uber',
    'Qualcomm', 'Salesforce', 'Stripe', 'Adobe', 'AECOM', 'NVIDIA', 'PayPal', 'Mastercard', 'Capgemini', 'Walmart', 'HP', 'CGI',
    'Tech Mahindra', 'Epic', 'Collins Aerospace', 'Bloomberg', 'U.S. Department of Veterns Affairs', 'BBC', 'Ericsson', 'State Farm',
    'Deloitte', 'Tesla', 'Chicago Public Schools', 'HSBC', 'Facebook', 'NYC Department of Education', 'ManTech', 'UniFirst Corporation',
    'ATI Physical Therapy', 'Deliveroo', 'Liberty Manual Insurance', 'Robert Half', 'DaVita Kidney Care', 'Allied Universal', 
    'Milliman', 'Allstate', 'Angi', 'Vodafone', 'Lloyds Banking Group', 'Fastenal', 'Citi', 'EY', 'Dell Technologies', 'Houlihan Lokey',
    'Creative Artists Agency', 'eXp Realty', 'The Home Depot', 'Grand Canyon Education, Inc.', 'Compunnel Inc.', 'US Navy', 'PepsiCo']
selected_company = company_list[random.randint(0,len(company_list)-1)]
company_code, _ = get_company_code(selected_company)
company_slr_coef = registered_coefs[(registered_coefs['feature'] == 'company') & \
        (registered_coefs['code'] == company_code)]['coef'].values[0]
### MAIN SKILL SALARY COEFFICIENT (0.0-1.0) (AVG & MAX)
main_skill_list = ['Financial Management', 'Python', 'Data Engineering', 'Microsoft Office', 'Programming', 'SQL', 'SQL Scripting',
    'Java', 'Microsoft Excel', 'Leadership', 'Data Analysis', 'C++', 'Marketing Research', 'Sales and Marketing', 'JavaScript',
    'Customer Service', 'Communication', 'HTML', 'Project Management', 'Research', 'C', 'PowerPoint', 'Machine Learning',
    'Microsoft Word', 'Software Development', 'HTML/CSS', 'Teamwork', 'Public Speaking', 'Sales Techniques', 'Linux', 'Sales Strategy',
    'Management', 'Matlab', 'Office Management', 'Records Management', 'Deadline Management', 'CSS', 'Marketing', 'Visual',
    'Social Media', 'Agile', 'Sales', 'Testing and Validation', 'Testing', 'Product Knowledge', 'git', 'R', 'MySQL', 'Problem Solving',
    'Social Media Platforms', 'AWS', 'Algorithms', 'Time Management', 'Sales Support', 'Computer Networks', 'Statistics',
    'Business Analysis', 'Analytical Skills', 'Tableau', 'Writing', 'User Interface Design', 'Data Science', 'DevOps', 'Databases',
    'Marketing Campaigns', 'Web Development', 'React', 'Marketing Knowledge', 'C#', 'Data Visualization', 'Training',
    'Object Oriented Programming', 'Data Structures', 'Data Mining', 'Photoshop', 'User Experience Design', 'Strategic Planning',
    'Node.js', 'Event Planning', 'Windows']
main_skills = ", ".join(np.random.choice(main_skill_list, ms_counts, replace=False))
main_skills_codes, _ = get_main_skills_code(main_skills)
main_skills_codes = main_skills_codes.strip().replace(' ',',')
main_skills_code_list = main_skills_codes.split(',')
main_skills_coefs = []
for x in range(len(main_skills_code_list)):
    main_skills_coefs.append(registered_coefs[(registered_coefs['feature'] == 'main_skill') & \
                   (registered_coefs['code'] == main_skills_code_list[x])]['coef'].values[0])
main_skill_slr_coef_avg = round(np.array(main_skills_coefs).mean(),3)
main_skill_slr_coef_max = round(np.array(main_skills_coefs).max(),3)

print("#### CUSTOM INPUT SUMMARY ####")
print("COMPANY SCORE =", company_score)
print("WORKING YEAR =", working_year)
print("EDUCATION SCORE =", education_score)
print("MAIN SKILL COUNTS =", ms_counts)
print("SKILL COUNTS =", skill_counts)
print("COUNTRY COEF = {} << {}".format(country_coef, selected_country))
print("TITLE SALARY COEF = {} << {} << {}".format(title_slr_coef, title_code, selected_title))
print("TITLE SENIORITY SALARY COEF = {} << {}".format(title_seniority_slr_coef, title_seniority_code))
print("POSITION SALARY COEF = {} << {} << {}".format(position_slr_coef, position_code, selected_position))
print("POSITON SENIORITY SALARY COEF = {} << {}".format(position_seniority_slr_coef, position_seniority_code))
print("EMPLOYMENT TYPE COEF = {} << {}".format(employment_type_coef, selected_emp_type))
print("EMPLOYMENT TYPE SALARY COEF = {}".format(employment_type_slr_coef))
print("HIGHEST EDUCATION DEGREE LEVEL = {} << {} << {}".format(highest_edu_degree_lvl, highest_edu_degree_name, edu_degrees))
print("EDUCATION DEGREE SALARY COEF = {}".format(edu_deg_slr_coef))
print("EDUCATION DEGREE MAJOR SALARY COEF (AVG) = {} << {} << {} << {}".format(
    edu_degree_major_slr_coef_avg, edu_degree_major_coefs, edu_degree_major_code_list, edu_degree_majors))
print("EDUCATION DEGREE MAJOR SALARY COEF (MAX) = {}".format(edu_degree_major_slr_coef_max))
print("COMPANY COEF = {} << {} << {}".format(company_slr_coef, company_code, selected_company))
print("MAIN SKILL SALARY COEF (AVG) = {} << {} << {} << {}".format(
    main_skill_slr_coef_avg, main_skills_coefs, main_skills_code_list, main_skills))
print("MAIN SKILL SALARY COEF (MAX) = {}".format(main_skill_slr_coef_max))

### Here are the numerical features of full custom input altogether
full_custom_input = np.array([company_score, working_year, education_score, ms_counts, skill_counts, country_coef,
                              title_slr_coef, title_seniority_slr_coef, position_slr_coef, position_seniority_slr_coef,
                              employment_type_coef, employment_type_slr_coef, highest_edu_degree_lvl, edu_deg_slr_coef, 
                              edu_degree_major_slr_coef_max, company_slr_coef, main_skill_slr_coef_avg])
print("\n#### FULL CUSTOM INPUT VALUES ####\n{}".format(full_custom_input))

#### CUSTOM INPUT SUMMARY ####
COMPANY SCORE = 8.7
WORKING YEAR = 37
EDUCATION SCORE = 7.9
MAIN SKILL COUNTS = 12
SKILL COUNTS = 50
COUNTRY COEF = 0.621 << Italy
TITLE SALARY COEF = 0.277 << BC24 << Insurance Agent
TITLE SENIORITY SALARY COEF = 0.488 << NO_SENIORITY
POSITION SALARY COEF = 0.477 << WC35 << Investment Banking Analyst
POSITON SENIORITY SALARY COEF = 0.518 << NO_SENIORITY
EMPLOYMENT TYPE COEF = 0.3 << Internship
EMPLOYMENT TYPE SALARY COEF = 0.692
HIGHEST EDUCATION DEGREE LEVEL = 1.0 << PHD << MASTERS,PHD,HIGH_SCHOOL
EDUCATION DEGREE SALARY COEF = 1.0
EDUCATION DEGREE MAJOR SALARY COEF (AVG) = 0.697 << [0.812, 0.854, 0.426] << ['EDM50', 'EDM54', 'EDM55'] << Urban Planning, Radio, Television and Film, Statistical Science
EDUCATION DEGREE MAJOR SALARY COEF (MAX) = 0.854
COMPANY COEF = 0.703 << CMP04 << Apple
MAIN SKILL SALARY COEF (AVG) = 0.806 << [0.767, 0.914, 0.605, 0.707, 0.691, 0.833, 0.814, 1.0, 0.728, 0.89, 0.867, 0.856] << ['MS12', 'MS24', 'MS30', 'MS34', 'MS37', 'MS

## Make Predictions with ML Regression Models

In [12]:
print("#### FULL CUSTOM INPUT VALUES ####")
print("{:>38} = {}".format("COMPANY SCORE", company_score))
print("{:>38} = {}".format("WORKING YEAR", working_year))
print("{:>38} = {}".format("EDUCATION SCORE", education_score))
print("{:>38} = {}".format("MAIN SKILL COUNT", ms_counts))
print("{:>38} = {}".format("SKILL COUNT", skill_counts))
print("{:>38} = {}".format("COUNTRY COEF", country_coef))
print("{:>38} = {}".format("TITLE SALARY COEF", title_slr_coef))
print("{:>38} = {}".format("TITLE SENIORITY SALARY COEF", title_seniority_slr_coef))
print("{:>38} = {}".format("POSITION SALARY COEF", position_slr_coef))
print("{:>38} = {}".format("POSITION SENIORITY SALARY COEF", position_seniority_slr_coef))
print("{:>38} = {}".format("EMPLOYMENT TYPE COEF", employment_type_coef))
print("{:>38} = {}".format("EMPLOYMENT TYPE SALARY COEF", employment_type_slr_coef))
print("{:>38} = {}".format("HIGHEST EDUCATION DEGREE LEVEL", highest_edu_degree_lvl))
print("{:>38} = {}".format("EDUCATION DEGREE SALARY COEF", edu_deg_slr_coef))
print("{:>38} = {}".format("EDUCATION DEGREE MAJOR SALARY COEF MAX", edu_degree_major_slr_coef_max))
print("{:>38} = {}".format("COMPANY SALARY COEF", company_slr_coef))
print("{:>38} = {}".format("MAIN SKILL SALARY COEF AVG", main_skill_slr_coef_avg))
full_custom_input_rs = full_custom_input.reshape(1,-1)

with open('ml_linreg_proto.pkl', 'rb') as ml_file:
    linreg = pickle.load(ml_file)
linreg_pred = linreg.predict(full_custom_input_rs)
with open('ml_rforest_reg_proto.pkl', 'rb') as ml_file:
    rforest = pickle.load(ml_file)
rforest_pred = rforest.predict(full_custom_input_rs)
with open('ml_gradboost_reg_proto.pkl', 'rb') as ml_file:
    gradBoost = pickle.load(ml_file)
gradBoost_pred = gradBoost.predict(full_custom_input_rs)
with open('ml_xgboost_reg_proto.pkl', 'rb') as ml_file:
    xgb = pickle.load(ml_file)
xgb_pred = xgb.predict(full_custom_input_rs)
pred_c1 = np.array([linreg_pred,rforest_pred,gradBoost_pred,xgb_pred]).mean()
pred_c2 = np.array([rforest_pred,gradBoost_pred,xgb_pred]).mean()

print("#### PREDICTION RESULTS ####")
print("LINEAR REG.        = {:14.6f}".format(linreg_pred[0]))
print("RANDOM FOREST REG. = {:14.6f}".format(rforest_pred[0]))
print("GRAD BOOST REG.    = {:14.6f}".format(gradBoost_pred[0]))
print("XGBOOST REG.       = {:14.6f}".format(xgb_pred[0]))
print("-"*36)
print("COMBINED ALL       = {:14.6f}".format(pred_c1))
print("COMBINED 2/3/4     = {:14.6f}".format(pred_c2))

#### FULL CUSTOM INPUT VALUES ####
                         COMPANY SCORE = 8.7
                          WORKING YEAR = 37
                       EDUCATION SCORE = 7.9
                      MAIN SKILL COUNT = 12
                           SKILL COUNT = 50
                          COUNTRY COEF = 0.621
                     TITLE SALARY COEF = 0.277
           TITLE SENIORITY SALARY COEF = 0.488
                  POSITION SALARY COEF = 0.477
        POSITION SENIORITY SALARY COEF = 0.518
                  EMPLOYMENT TYPE COEF = 0.3
           EMPLOYMENT TYPE SALARY COEF = 0.692
        HIGHEST EDUCATION DEGREE LEVEL = 1.0
          EDUCATION DEGREE SALARY COEF = 1.0
EDUCATION DEGREE MAJOR SALARY COEF MAX = 0.854
                   COMPANY SALARY COEF = 0.703
            MAIN SKILL SALARY COEF AVG = 0.806
#### PREDICTION RESULTS ####
LINEAR REG.        =  148919.049063
RANDOM FOREST REG. =  143182.740000
GRAD BOOST REG.    =  128338.346671
XGBOOST REG.       =  167181.578125
------------